# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR\^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset follows the [Croissant schema](https://mlcommons.org/croissant/) and is described at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed. If already installed, this step is a no-op.
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the URL to the Croissant JSON-LD schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset object
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's review which record sets, fields, and field IDs are available in the dataset.

In [ ]:
# List all record sets with their @id
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- @id: {record_set.id}  (name: {record_set.name})")

# For this dataset, let's inspect the fields of the main record set.
main_record_set_id = dataset.record_sets[0].id
print(f"\nFields for record set {main_record_set_id}:")
for field in dataset.record_sets[0].fields:
    print(f"  - @id: {field.id} (name: {field.name}, dataType: {field.data_type})")

## 3. Data Extraction
We extract data from a record set. We will use the record set and field `@id`s from the previous overview step.

In [ ]:
# For demonstration, we'll load the first available record set
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
// Additional print just to show the columns for reference
        print(f"Loaded columns for {rs_id}:", df.columns.tolist())
        display(df.head())

# For further processing, use the first DataFrame (the main tabular data)
main_df_id = record_set_ids[0]
main_df = dataframes[main_df_id]
print("\nMain DataFrame Columns:", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze key statistics from the data. We'll filter, normalize, and group using only field `@id`s.

For demonstration, let's assume one of the numeric variables is `https://api.app.sen.science/frontiers/7862866/age_at_second_crc` (the true @id should be obtained from the record set fields listed above, and must be replaced if different). We'll group by a categorical variable `https://api.app.sen.science/frontiers/7862866/gender` if present.

In [ ]:
# Replace these @ids with the true IDs from the overview above as needed
numeric_field_id = None
# Try to automatically identify a likely numeric field
for col in main_df.columns:
    if "age" in col.lower() or "year" in col.lower() or "interval" in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes('number').columns[0]  # Fallback to first numeric column

print(f"Using numeric field: {numeric_field_id}")

# Filtering records where the field is above a threshold (e.g., age > 50)
thresh = 50
filtered_df = main_df[main_df[numeric_field_id] > thresh].copy()
print(f"Filtered records where {numeric_field_id} > {thresh} (count: {len(filtered_df)}):")
display(filtered_df.head())

# Normalize the chosen numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping example (by anatomical location, if present)
group_field_id = None
for col in main_df.columns:
    if "anatom" in col.lower() or "gender" in col.lower() or "sex" in col.lower():
        group_field_id = col
        break
if group_field_id:
    print(f"\nGrouping filtered data by {group_field_id} and calculating mean:")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    display(grouped_df)

## 5. Visualization
Let's visualize some key distributions or relationships between variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8, 6))
sns.histplot(main_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping field exists, show boxplot vs group
if group_field_id:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} Distribution by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR\^2 dataset using the Croissant schema and `mlcroissant`
- Explored metadata and reviewed record sets and field `@id`s
- Loaded the main record set into a DataFrame using exact `@id` references
- Conducted simple EDA: filtered and normalized a key numeric field, and grouped by another field
- Visualized distributions using histograms and boxplots

This approach shows how `mlcroissant` enables FAIR data access and direct, identifier-driven data handling for reliable, reproducible ML workflows.